# How to Use the Statistics API

The `/statistics` endpoint can be used to get summary statistics for a geojson `Feature` or `FeatureCollection`.

This notebook demonstrates how to use the statistics API using the [TROPESS Chemical Reanalysis O3 Monthly 3-dimensional Product](https://cmr.earthdata.nasa.gov/search/concepts/C2837626477-GES_DISC) dataset.

## Setup

In [23]:
import json
from datetime import datetime, timezone

import httpx

# titiler_endpoint = "https://staging.openveda.cloud/api/titiler-cmr"  # staging endpoint
titiler_endpoint = (
    "https://v4jec6i5c0.execute-api.us-west-2.amazonaws.com"  # dev endpoint
)

Specifiy a geojson dictionary to use in the request.

In [24]:
geojson_dict = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {},
            "geometry": {
                "coordinates": [
                    [
                        [-20.79973248834736, 83.55979308678764],
                        [-20.79973248834736, 75.0115425216471],
                        [14.483337068956956, 75.0115425216471],
                        [14.483337068956956, 83.55979308678764],
                        [-20.79973248834736, 83.55979308678764],
                    ]
                ],
                "type": "Polygon",
            },
        }
    ],
}

TiTiler-CMR will use one or more `sel` parameter values to select specific subsets from a multidimensional data granule. The TROPESS dataset consists of monthly granules with daily arrays. To load the values from 2021-10-10 we provide that timestamp to the `temporal` parameter then provide `sel=time=nearest::{datetime}` to have TiTiler-CMR interpolate the value you provided to the `temporal` parameter. For fuzzy matching you can provide the `nearest::` prefix to the value.

In [25]:
r = httpx.post(
    f"{titiler_endpoint}/xarray/statistics",
    params=(
        ("collection_concept_id", "C2837626477-GES_DISC"),
        # Timestamp or temporal range for CMR granule query
        ("temporal", datetime(2021, 10, 10, tzinfo=timezone.utc).isoformat()),
        # xarray backend query parameters
        ("variables", "o3"),
        ("sel", "time=nearest::{datetime}"),  # gets interpolated
        ("sel", "lev=1000"),
    ),
    json=geojson_dict,
    timeout=None,
).json()

print(json.dumps(r, indent=2))

{
  "detail": "Error parsing datetime string \"{datetime}\" at position 0"
}


You can chose a different time slice from the same granule simply by updating the `datetime` query parameter.

In [21]:
r = httpx.post(
    f"{titiler_endpoint}/xarray/statistics",
    params=(
        ("collection_concept_id", "C2837626477-GES_DISC"),
        # Datetime for CMR granule query
        ("temporal", datetime(2021, 12, 10, tzinfo=timezone.utc).isoformat()),
        # xarray backend query parameters
        ("variables", "o3"),
        ("sel", "time=nearest::{datetime}"),  #
        ("sel", "lev=1000"),
    ),
    json=geojson_dict,
    timeout=None,
).json()

print(json.dumps(r, indent=2))

{
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "geometry": {
        "type": "Polygon",
        "coordinates": [
          [
            [
              -20.79973248834736,
              83.55979308678764
            ],
            [
              -20.79973248834736,
              75.0115425216471
            ],
            [
              14.483337068956956,
              75.0115425216471
            ],
            [
              14.483337068956956,
              83.55979308678764
            ],
            [
              -20.79973248834736,
              83.55979308678764
            ]
          ]
        ]
      },
      "properties": {
        "statistics": {
          "b1": {
            "min": 18.230709075927734,
            "max": 37.22050476074219,
            "mean": 27.433626174926758,
            "count": 192.0,
            "sum": 5267.25634765625,
            "std": 4.652167252044063,
            "median": 27.250431060791016,
       